# Описание датасета

Датасет подготовлен для учебной задачи Entity Resolution: необходимо определить, какие профили относятся к одному реальному человеку.

Одна строка датасета соответствует событию профиля в маркетплейсе скидок. Один и тот же профиль может встречаться в нескольких строках.

## Объём данных

В наборе 187270 уникальных профилей, соответствующих исходным `customer_id`. Из них 2654 профиля входят в `multi_profile` entity, то есть связаны с другими профилями как один реальный человек.

## Разбиение

Данные разделены на три набора:

- `train` - 60%
- `test` - 20%
- `validation` - 20%

Разбиение выполнено с учётом типа entity: `multi_profile` и `single_profile` разбивались на наборы отдельно. Все записи одной entity попадают только в один набор.


## Ключевые сущности

| Поле | Интерпретация |
|---|---|
| `profile_id` | Идентификатор профиля. |
| `entity_id` | Идентификатор реального пользователя. Несколько разных `profile_id` с одним `entity_id` считаются профилями одного пользователя. |
| `entity_type` | Тип entity: `single_profile`, если entity содержит один профиль, и `multi_profile`, если в одну entity входит несколько профилей. |

## Состав полей

| Поле | Тип | Описание |
|---|---|---|
| `created_at` | timestamp | Время события. Может использоваться для сортировки событий и построения временных признаков. |
| `first_name` | string | Анонимизированное имя. Преобразование детерминированное: одно исходное имя преобразуется одинаково. Род имени сохранён. |
| `last_name` | string | Анонимизированная фамилия. Преобразование детерминированное, род фамилии сохранён. |
| `email` | string | Анонимизированный email. Локальная часть до `@` заменена синтетически, домен оставлен без изменений. Это позволяет использовать признаки домена. |
| `phone` | string | Анонимизированный телефон. Сохранены код страны и первые 3 цифры кода оператора. Остальные цифры заменены детерминированно. |
| `birthday` | date | Анонимизированная дата рождения. Даты 1 января оставлены без изменения. |
| `sex` | string | Пол из исходного профиля, если он был указан. Может быть пустым. |
| `non_processing_features` | array<string> | Набор технических и контекстных признаков события в формате строк. Например: устройство, браузер, ОС, географические признаки. |
| `realtime_features` | string | JSON-строка с признаками, рассчитанными в момент события. Например: количество визитов, страна, город, локальный час, день недели. |
| `fs_features` | array<string> | Исторические признаки профиля из feature store. Формат элемента: `feature_name:value` или флаг без значения, например `is_gmail`. |
| `profile_id` | string | Идентификатор профиля. |
| `entity_id` | string | Идентификатор entity. |
| `entity_type` | string | Тип entity: `single_profile` или `multi_profile`. |

`entity_id` является целевой разметкой для задачи Entity Resolution.

## Как читать `fs_features`

`fs_features` - это массив строк. Каждый элемент описывает один бинарный или категориальный признак.

Примеры:

- `visited_365:1640` - профиль посещал сайт с замаскированным id `1640` за последние 365 дней.
- `visited_30:3130` - профиль посещал сайт с замаскированным id `3130` за последние 30 дней.
- `source_site_365:3768` - за последние 365 дней у профиля встречался source site с замаскированным id `3768`.
- `has_account:2046` - у профиля есть аккаунт на сайте с замаскированным id `2046`.
- `is_man` - флаг без дополнительного значения.
- `postman_response_90:ok` - категориальный признак email-коммуникации за последние 90 дней.

Числовые значения в site-признаках являются замаскированными site id.

## Site-id признаки в `fs_features`

У следующих признаков значение после `:` является замаскированным site id:

- `has_accept_30`
- `has_accept_365`
- `has_account`
- `has_click_30`
- `has_click_365`
- `has_order_30`
- `has_order_365`
- `has_view_90`
- `source_site_30`
- `source_site_365`
- `visited_30`
- `visited_365`

Интерпретация суффиксов:

- `_30` - признак рассчитан за последние 30 дней.
- `_90` - признак рассчитан за последние 90 дней.
- `_365` - признак рассчитан за последние 365 дней.

## Остальные признаки в `fs_features`

Флаги:

- `is_gmail`
- `is_man`
- `is_phone`
- `is_woman`
- `is_yandex`
- `was_phone_lead`

Категориальные признаки:

- `mm_event_30`, `mm_event_90`
- `postman_action_30`, `postman_action_90`
- `postman_campaign_30`, `postman_campaign_90`
- `postman_response_30`, `postman_response_90`
- `postman_scenario_30`, `postman_scenario_90`

## Анонимизация

Выполнена детерминированная анонимизация следующих полей:

- `profile_id`
- `first_name`
- `last_name`
- `email`
- `phone`
- `birthday`
- site id внутри `fs_features`

Детерминированность означает, что одно и то же исходное значение преобразуется в одно и то же синтетическое значение. Это важно для задачи Entity Resolution: сохраняется возможность сопоставлять одинаковые значения между строками, но исходные персональные данные не раскрываются.

Особенности анонимизации:

- Для имён и фамилий сохранён род.
- Для email сохранён домен.
- Для телефона сохранены код страны и первые 3 цифры кода оператора.



In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path(os.path.abspath('..'))
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATA_DIR = PROJECT_ROOT / 'data'

In [ ]:
df = pd.read_parquet(DATA_DIR / 'split_label_train_V3.snappy.parquet')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

last_name почти пустой (1621 из 68036 → ~2.4%) — бесполезен

birthday практически пустой (407 из 68036 → ~0.6%) — бесполезен

phone очень редкий (4966 → ~7.3%)

first_name заполнен у 27% (18479/68036)

email заполнен у 100% — важнейший признак!

sex почти полный (99.9%) — отлично

Есть пропуски в non_processing_features, realtime_features, fs_features (241 строка без них)

In [ ]:
# Сколько всего уникальных entity?
print(f"entity_id: {df['entity_id'].nunique()/len(df)}")
print(f" profile_id: {df['profile_id'].nunique()/len(df)}")

In [ ]:
#Есть ли профили, которые встречаются несколько раз?
profile_counts = df['profile_id'].value_counts()
print(f"Профилей с несколькими событиями: {(profile_counts > 1).sum()}")
print(profile_counts.value_counts().sort_index())

In [ ]:
#Есть ли профили, которые встречаются несколько раз?
entity_counts = df['entity_id'].value_counts()
print(f"Профилей с несколькими событиями: {(entity_counts > 1).sum()}")
print(entity_counts.value_counts().sort_index())

## Анализ multi_profile

In [ ]:
entity_profile_counts = df.groupby('entity_id')['profile_id'].nunique()
multi_entities = entity_profile_counts[entity_profile_counts > 1].index
single_entities = entity_profile_counts[entity_profile_counts == 1].index

In [ ]:
df['is_multi_entity'] = df['entity_id'].isin(multi_entities)

In [ ]:
fields = ['first_name', 'last_name', 'email', 'phone', 'birthday', 'sex']
results = []

for field in fields:
    single_filled = df[~df['is_multi_entity']][field].notna().mean()
    multi_filled = df[df['is_multi_entity']][field].notna().mean()
    results.append({
        'field': field,
        'single_entity': f"{single_filled*100:.2f}%",
        'multi_entity': f"{multi_filled*100:.2f}%",
        'difference': f"{(multi_filled - single_filled)*100:+.2f}%"
    })

pd.DataFrame(results)

### Анализ email

In [ ]:
# Топ доменов для single vs multi
print(df[~df['is_multi_entity']]['email'].value_counts().head(10))

print(df[df['is_multi_entity']]['email'].value_counts().head(10))

# Анализ уникальности доменов внутри multi entity
def analyze_domains_in_multi(df, multi_entities):
    multi_df = df[df['entity_id'].isin(multi_entities)]
    email_stats = []

    for entity_id in multi_entities[:100]:  # первые 100 для скорости
        entity_data = df[df['entity_id'] == entity_id]
        emails = entity_data['email'].unique()
        email_stats.append({
            'entity_id': entity_id,
            'n_profiles': entity_data['profile_id'].nunique(),
            'n_unique_email': len(emails),
            'all_same_email': len(emails) == 1
        })

    email_df = pd.DataFrame(email_stats)
    print(f"Среднее количество уникальных email на multi entity: {email_df['n_unique_email'].mean():.2f}")

analyze_domains_in_multi(df, multi_entities)

### Анализ доменов

In [ ]:
# Извлекаем домены
df['email_domain'] = df['email'].str.split('@').str[1]

print(df[~df['is_multi_entity']]['email_domain'].value_counts().head(10))

print(df[df['is_multi_entity']]['email_domain'].value_counts().head(10))

# Анализ уникальности доменов внутри multi entity
def analyze_domains_in_multi(df, multi_entities):
    multi_df = df[df['entity_id'].isin(multi_entities)]
    domain_stats = []

    for entity_id in multi_entities[:100]:  # первые 100 для скорости
        entity_data = df[df['entity_id'] == entity_id]
        domains = entity_data['email_domain'].unique()
        domain_stats.append({
            'entity_id': entity_id,
            'n_profiles': entity_data['profile_id'].nunique(),
            'n_unique_domains': len(domains),
            'all_same_domain': len(domains) == 1
        })

    domain_df = pd.DataFrame(domain_stats)
    print(f"\nВ {domain_df['all_same_domain'].mean()*100:.1f}% multi entity все email с ОДИНАКОВЫМ доменом")
    print(f"Среднее количество уникальных доменов на multi entity: {domain_df['n_unique_domains'].mean():.2f}")

analyze_domains_in_multi(df, multi_entities)

In [ ]:
# Какие имена чаще всего встречаются в multi?
name_multi_ratio = df.groupby('first_name').agg({
    'is_multi_entity': 'mean',
    'profile_id': 'count'
}).rename(columns={'is_multi_entity': 'multi_ratio', 'profile_id': 'frequency'})

# Имена с высоким multi_ratio (часто в multi entity)
high_multi_names = name_multi_ratio[
    (name_multi_ratio['multi_ratio'] > 0.5) &
    (name_multi_ratio['frequency'] > 3)
].sort_values('multi_ratio', ascending=False)

print("Имена, которые в >70% случаев принадлежат multi entity:")
print(high_multi_names.head(15))

# Имена, которые ТОЛЬКО в single
only_single_names = name_multi_ratio[
    (name_multi_ratio['multi_ratio'] == 0) &
    (name_multi_ratio['frequency'] > 0)
]
print(f"\nИмена, которые встречаются ТОЛЬКО в single entity: {len(only_single_names)}")

In [ ]:
# Распределение пола
sex_distribution = pd.crosstab(df['is_multi_entity'], df['sex'], normalize='index')
print("Распределение пола по типу entity:")
print(sex_distribution)

# Конфликты пола внутри multi entity
def has_sex_conflict(entity_group):
    sexes = entity_group['sex'].dropna().unique()
    return len(sexes) > 1

multi_df = df[df['is_multi_entity']]
sex_conflicts = multi_df.groupby('entity_id').apply(has_sex_conflict)

print(f"\nEntity с конфликтом полов: {sex_conflicts.sum()} из {len(multi_entities)} ({sex_conflicts.mean()*100:.2f}%)")

if sex_conflicts.sum() > 0:
    print("\nПример конфликта:")
    conflict_id = sex_conflicts[sex_conflicts].index[0]
    print(df[df['entity_id'] == conflict_id][['profile_id', 'first_name', 'sex', 'email_domain']])

In [ ]:
print(f"Всего записей с телефоном: {df['phone'].notna().sum():,} ({df['phone'].notna().mean()*100:.2f}%)")
print(f"Из них в multi entity: {(df[df['is_multi_entity']]['phone'].notna().sum()):,}")

# Телефоны, привязанные к нескольким profile_id
phone_to_profiles = df[df['phone'].notna()].groupby('phone')['profile_id'].nunique()
phones_with_multiple = phone_to_profiles[phone_to_profiles > 1]

print(f"\nТелефонов, связанных с >1 profile_id: {len(phones_with_multiple)}")

In [ ]:
# Когда создаются профили (час дня)
df['hour'] = df['created_at'].dt.hour
df['day_of_week'] = df['created_at'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# По часам
axes[0].hist(df[~df['is_multi_entity']]['hour'], bins=24, alpha=0.5, label='Single', density=True)
axes[0].hist(df[df['is_multi_entity']]['hour'], bins=24, alpha=0.5, label='Multi', density=True)
axes[0].set_xlabel('Час дня')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Распределение событий по часам')
axes[0].legend()

# По дням недели
axes[1].hist(df[~df['is_multi_entity']]['day_of_week'], bins=7, alpha=0.5, label='Single', density=True)
axes[1].hist(df[df['is_multi_entity']]['day_of_week'], bins=7, alpha=0.5, label='Multi', density=True)
axes[1].set_xlabel('День недели (0=пн)')
axes[1].set_ylabel('Плотность')
axes[1].set_title('Распределение событий по дням недели')
axes[1].legend()

plt.tight_layout()
plt.show()

# Временная близость профилей в multi entity
def get_time_span(entity_group):
    times = entity_group['created_at']
    if len(times) < 2:
        return None
    return (times.max() - times.min()).total_seconds() / 3600  # в часах

time_spans = multi_df.groupby('entity_id').apply(get_time_span).dropna()

print(f"\nВременной разброс между первым и последним событием в multi entity (часы):")
print(time_spans.describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))

# Процент entity, где профили созданы в течение 1 часа
within_hour = (time_spans <= 1).mean()
print(f"\nПрофили созданы в течение 1 часа: {within_hour*100:.1f}%")

# Обработка признаков

## Парсинг fs_features, non_processing_features, realtime_features

In [ ]:
def safe_convert(x):
    if x is None:
        return {}

    result = {}
    for item in x:
        if ':' in str(item):
            parts = str(item).split(':', 1)  # split only on first colon
            if len(parts) == 2:
                key, value = parts
                result[key.strip()] = value.strip()
            else:
                # Если все еще не 2 части, используем как ключ
                result[str(item)] = None
        else:
            # Нет двоеточия - используем весь элемент как ключ
            result[str(item)] = None

    return result

df['fs_features'] = df['fs_features'].apply(safe_convert)
df['non_processing_features'] = df['non_processing_features'].apply(safe_convert)

In [ ]:
df['realtime_features'] = df['realtime_features'].apply(lambda x: {} if pd.isna(x) else x)

# Затем применяем вашу функцию (уже без NaN)
def json_convert(x):
    if isinstance(x, str):
        return json.loads(x) if x else {}
    return x if isinstance(x, dict) else {}

df['realtime_features'] = df['realtime_features'].apply(json_convert)

In [ ]:
def expand_all_features(df):
    """
    Раскрывает все словари в non_processing_features, realtime_features, fs_features
    """
    df = df.copy()

    # 1. Раскрываем non_processing_features
    # Извлекаем все ключи
    np_keys = set()
    for d in df['non_processing_features'].dropna():
        np_keys.update(d.keys())
    print(f"   Найдено ключей: {len(np_keys)}")
    print(f"   Ключи: {np_keys}")

    # Создаем колонки для каждого признака
    for key in np_keys:
        col_name = f'np_{key}'
        df[col_name] = df['non_processing_features'].apply(
            lambda x: x.get(key) if isinstance(x, dict) else None
        )

    # Переименовываем основные признаки для удобства
    rename_map = {
        'np_device': 'device',
        'np_osfamily': 'osfamily',
        'np_browser': 'browser',
        'np_geoname_id': 'geoname_id',
        'np_subdivision_1_iso_code': 'region_code'
    }

    for old, new in rename_map.items():
        if old in df.columns:
            df[new] = df[old]
            df.drop(columns=[old], inplace=True)

    # 2. Раскрываем realtime_features
    # Извлекаем все ключи
    rt_keys = set()
    for d in df['realtime_features'].dropna():
        rt_keys.update(d.keys())
    print(f"   Найдено ключей: {len(rt_keys)}")
    print(f"   Ключи: {rt_keys}")

    # Создаем колонки для каждого признака
    for key in rt_keys:
        col_name = f'rt_{key}'
        df[col_name] = df['realtime_features'].apply(
            lambda x: x.get(key) if isinstance(x, dict) else None
        )

    # Переименовываем для удобства
    rename_map_rt = {
        'rt_country': 'country',
        'rt_is_million': 'is_million_city',
        'rt_tz_offset': 'tz_offset',
        'rt_geoname': 'city_name',
        'rt_geoid': 'city_geoid',
        'rt_local_hour': 'local_hour',
        'rt_day': 'day_of_week_from_rt',
        'rt_population': 'city_population',
        'rt_visit_count': 'visit_count'
    }

    for old, new in rename_map_rt.items():
        if old in df.columns:
            df[new] = df[old]
            df.drop(columns=[old], inplace=True)

    # 3. Раскрываем fs_features
    # Собираем все уникальные ключи
    fs_keys = set()
    for d in df['fs_features'].dropna():
        fs_keys.update(d.keys())
    print(f"   Найдено ключей: {len(fs_keys)}")

    # Разделяем ключи по типам
    site_id_prefixes = {
        'has_accept_30', 'has_accept_365', 'has_account',
        'has_click_30', 'has_click_365', 'has_order_30',
        'has_order_365', 'has_view_90', 'source_site_30',
        'source_site_365', 'visited_30', 'visited_365'
    }

    flags = {'is_gmail', 'is_man', 'is_phone', 'is_woman',
             'is_yandex', 'was_phone_lead'}

    postman_prefixes = {'mm_event', 'postman_action', 'postman_campaign',
                        'postman_response', 'postman_scenario'}

    site_id_keys = []
    flag_keys = []
    postman_keys = []
    other_fs_keys = []

    for key in fs_keys:
        # Проверяем site-id признаки (ключ:значение)
        if key in site_id_prefixes:
            site_id_keys.append(key)
        # Проверяем флаги
        elif key in flags:
            flag_keys.append(key)
        # Проверяем postman признаки
        elif any(key.startswith(prefix) for prefix in postman_prefixes):
            postman_keys.append(key)
        else:
            other_fs_keys.append(key)

    print(f"   Site-id признаков: {len(site_id_keys)}")
    print(f"   Флагов: {len(flag_keys)}")
    print(f"   Postman признаков: {len(postman_keys)}")
    print(f"   Прочих признаков: {len(other_fs_keys)}")

    # Раскрываем site-id признаки (сохраняем site_id как значение)
    for key in site_id_keys:
        df[f'fs_{key}'] = df['fs_features'].apply(
            lambda x: x.get(key) if isinstance(x, dict) else None
        )

    # Раскрываем флаги (True/False)
    for key in flag_keys:
        df[f'fs_{key}'] = df['fs_features'].apply(
            lambda x: key in x if isinstance(x, dict) else False
        )

    # Раскрываем postman признаки
    for key in postman_keys:
        df[f'fs_{key}'] = df['fs_features'].apply(
            lambda x: x.get(key) if isinstance(x, dict) else None
        )

    # Раскрываем прочие признаки
    for key in other_fs_keys:
        df[f'fs_{key}'] = df['fs_features'].apply(
            lambda x: x.get(key) if isinstance(x, dict) else None
        )

    return df

df = expand_all_features(df)

In [ ]:
import pandas as pd
import numpy as np


def build_full_profile_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Агрегированная таблица профилей:
    одна строка на profile_id со ВСЕМИ перечисленными признаками.
    """
    # Сортируем по времени, чтобы "last" действительно было последним событием
    df_sorted = df.sort_values("created_at")

    # Последнее непустое значение
    def last_notna(x):
        notna_vals = x.dropna()
        return notna_vals.iloc[-1] if len(notna_vals) > 0 else np.nan

    # Уникальные значения в список
    def unique_list(x):
        vals = x.dropna().unique()
        return list(vals) if len(vals) > 0 else []

    agg_dict = {
        # базовые
        "first_name": last_notna,
        "last_name": last_notna,
        "email": last_notna,
        "phone": last_notna,
        "birthday": last_notna,
        "sex": last_notna,
        "entity_id": last_notna,
        "is_multi_entity": last_notna,

        # время
        "created_at": ["min", "max"],
        "hour": unique_list,
        "day_of_week": unique_list,

        # non_processing / realtime (уже разложенные признаки)
        "np_is_not_russia": last_notna,
        "device": unique_list,
        "osfamily": unique_list,
        "browser": unique_list,
        "geoname_id": unique_list,
        "region_code": unique_list,
        "country": unique_list,
        "is_million_city": "any",
        "tz_offset": ["mean", "nunique"],
        "city_name": unique_list,
        "city_geoid": unique_list,
        "local_hour": ["mean", unique_list],
        "day_of_week_from_rt": unique_list,
        "city_population": "max",
        "visit_count": lambda x: x.dropna().max() if len(x.dropna()) > 0 else 0,

        # fs site-id признаки (все как списки уникальных site_id)
        "fs_has_order_365": unique_list,
        "fs_has_click_365": unique_list,
        "fs_visited_30": unique_list,
        "fs_has_accept_30": unique_list,
        "fs_source_site_365": unique_list,
        "fs_source_site_30": unique_list,
        "fs_has_order_30": unique_list,
        "fs_visited_365": unique_list,
        "fs_has_view_90": unique_list,
        "fs_has_click_30": unique_list,
        "fs_has_account": unique_list,
        "fs_has_accept_365": unique_list,

        # fs флаги
        "fs_is_yandex": "any",
        "fs_is_man": "any",
        "fs_was_phone_lead": "any",
        "fs_is_phone": "any",
        "fs_is_gmail": "any",
        "fs_is_woman": "any",

        # postman/mm (категориальные, берём все уникальные значения профиля)
        "fs_mm_event_30": unique_list,
        "fs_postman_response_30": unique_list,
        "fs_postman_campaign_90": unique_list,
        "fs_postman_response_90": unique_list,
        "fs_postman_action_90": unique_list,
        "fs_postman_action_30": unique_list,
        "fs_postman_campaign_30": unique_list,
        "fs_postman_scenario_30": unique_list,
        "fs_postman_scenario_90": unique_list,
        "fs_mm_event_90": unique_list,
    }

    # Вдруг каких-то колонок нет в df (robustness)
    agg_dict = {k: v for k, v in agg_dict.items() if k in df_sorted.columns}

    print("Агрегируем профили...")
    profiles = df_sorted.groupby("profile_id").agg(agg_dict)

    # Преобразуем multiindex колонок
    profiles.columns = [
        f"{c[0]}_{c[1]}" if isinstance(c, tuple) else c
        for c in profiles.columns
    ]

    # Переименования для удобства
    rename_map = {
        "created_at_min": "first_seen",
        "created_at_max": "last_seen",
        "tz_offset_mean": "avg_tz_offset",
        "tz_offset_nunique": "nunique_tz",
        "local_hour_mean": "avg_local_hour",
        "local_hour_<lambda>": "local_hours_list",
    }
    profiles = profiles.rename(columns={k: v for k, v in rename_map.items() if k in profiles.columns})

    profiles = profiles.reset_index()

    # email_domain / phone_prefix на уровне профиля
    if "email_last_notna" in profiles.columns:
        email_col = "email_last_notna"
    elif "email" in profiles.columns:
        email_col = "email"
    else:
        email_col = None

    if email_col:
        profiles["email_domain"] = (
            profiles[email_col]
            .fillna("")
            .str.split("@")
            .str[-1]
            .replace("", np.nan)
        )

    if "phone_last_notna" in profiles.columns:
        phone_col = "phone_last_notna"
    elif "phone" in profiles.columns:
        phone_col = "phone"
    else:
        phone_col = None

    if phone_col:
        profiles["phone_prefix"] = (
            profiles[phone_col]
            .fillna("")
            .str[:6]
            .replace("", np.nan)
        )

    return profiles


profiles = build_full_profile_table(df)
profiles.head(3).T

In [ ]:
df.to_csv(ARTIFACTS_DIR / 'df.csv', index=False)

In [ ]:
profiles.to_csv(ARTIFACTS_DIR / 'profiles.csv', index=False)

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split


entity_df = (
    df.groupby('entity_id', as_index=False)
      .agg(
          row_count=('entity_id', 'size'),
          is_multi_entity=('is_multi_entity', 'first')
      )
)

# 2. Делим entity_id на train+val и test
train_val_entity_df, test_entity_df = train_test_split(
    entity_df,
    test_size=0.20,
    random_state=42,
    stratify=entity_df['is_multi_entity']
)

# 3. Делим train+val на train и val
train_entity_df, val_entity_df = train_test_split(
    train_val_entity_df,
    test_size=0.25,   # 0.25 от 80% = 20% от общего объема
    random_state=42,
    stratify=train_val_entity_df['is_multi_entity']
)

# 4. Множества entity_id по сплитам
train_entities = set(train_entity_df['entity_id'])
val_entities = set(val_entity_df['entity_id'])
test_entities = set(test_entity_df['entity_id'])

# 5. Проставляем split всем строкам исходного df
df = df.copy()
df['split'] = np.select(
    [
        df['entity_id'].isin(train_entities),
        df['entity_id'].isin(val_entities),
        df['entity_id'].isin(test_entities),
    ],
    [
        'train',
        'valid',
        'test'
    ],
    default='unknown'
)

In [ ]:
export_map_for_test = pd.DataFrame()

In [ ]:
export_map_for_test[['split', 'entity_id']] = df[['split', 'entity_id']].copy()

In [ ]:
export_map_for_test.to_csv(ARTIFACTS_DIR / 'export_map_for_test.csv', index=False)

## Все доступные признаки в датасете

На основе вашего описания и выгруженных данных, вот полный список всех признаков, которые у вас есть для Entity Resolution:

---

### 1. **Основные демографические признаки**

| Признак | Тип | Описание | Использование в ER |
|---------|-----|----------|-------------------|
| `first_name` | string | Анонимизированное имя (род сохранён) | Сравнение имён |
| `last_name` | string | Анонимизированная фамилия (род сохранён) | Сравнение фамилий |
| `sex` | string | Пол (male/female/unknown) | Совпадение пола |
| `birthday` | date | Анонимизированная дата рождения (1 января не анонимизированы) | Совпадение даты рождения |
| `email` | string | Анонимизированный email (домен сохранён) | Точное совпадение, домен |
| `phone` | string | Анонимизированный телефон (код страны + код оператора сохранены) | Точное совпадение |

---

### 2. **Временные признаки**

| Признак | Тип | Описание | Использование в ER |
|---------|-----|----------|-------------------|
| `created_at` | timestamp | Время события | Сортировка, временные интервалы между событиями |
| `hour` | int (производный) | Час события (0-23) | Паттерны активности |
| `day_of_week` | int (производный) | День недели (0-6) | Паттерны активности |

---

### 3. **Признаки из `non_processing_features`**

Это массив строк с техническими и контекстными признаками. Уникальные значения:

| Признак | Тип | Уникальные значения | Использование в ER |
|---------|-----|---------------------|-------------------|
| `device` | string | `pc`, `smartphone`, `tablet` | Совпадение устройства |
| `osfamily` | string | `android`, `ios`, `linux`, `os-x`, `windows` | Совпадение ОС |
| `browser` | string | `chrome`, `firefox`, `opera`, `safari`, `yandex` | Совпадение браузера |
| `geoname_id` | int | 500+ значений (ID города в GeoNames) | Совпадение города |
| `subdivision_1_iso_code` | string | Множество кодов (ROS, MOS, SP, 01, 02, ...) | Совпадение региона |
| `is_not_russia` | bool | `None` (всегда None, т.е. все из России) | Не используется |

---

### 4. **Признаки из `realtime_features`**

JSON-строка с признаками, рассчитанными в момент события. Уникальные значения:

| Признак | Тип | Уникальные значения | Использование в ER |
|---------|-----|---------------------|-------------------|
| `country` | string | 113 стран (AE, AL, AM, ... RU, US, ...) | Совпадение страны |
| `is_million` | bool | `True`, `False` | Признак города-миллионника |
| `tz_offset` | int | -9 до +12 (целые часы) | Часовой пояс |
| `geoname` | string | 500+ названий городов | Совпадение города |
| `geoid` | int | 500+ ID городов GeoNames | Совпадение города |
| `local_hour` | int | 0-23 | Временные паттерны |
| `day` | int | 0-6 (день недели) | Временные паттерны |
| `population` | int | 500 до 10M+ | Размер города |
| `visit_count` | string | '0', '1', ... '226' | Количество визитов |

---

### 5. **Признаки из `fs_features` (Feature Store)**

Массив строк вида `feature_name:value` или флаги `feature_name`. Содержит:

#### A. **Site-id признаки (содержат маскированный site-id)**

| Признак | Значение |
|---------|----------|
| `has_accept_30:XXXX` | Принял предложение за 30 дней |
| `has_accept_365:XXXX` | Принял предложение за 365 дней |
| `has_account:XXXX` | Есть аккаунт на сайте |
| `has_click_30:XXXX` | Кликнул за 30 дней |
| `has_click_365:XXXX` | Кликнул за 365 дней |
| `has_order_30:XXXX` | Сделал заказ за 30 дней |
| `has_order_365:XXXX` | Сделал заказ за 365 дней |
| `has_view_90:XXXX` | Просматривал за 90 дней |
| `source_site_30:XXXX` | Источник трафика за 30 дней |
| `source_site_365:XXXX` | Источник трафика за 365 дней |
| `visited_30:XXXX` | Посещал за 30 дней |
| `visited_365:XXXX` | Посещал за 365 дней |

#### B. **Флаги (без значения)**

| Признак | Описание |
|---------|----------|
| `is_gmail` | Использует Gmail |
| `is_man` | Мужской пол |
| `is_phone` | Номер телефона подтверждён |
| `is_woman` | Женский пол |
| `is_yandex` | Использует Яндекс.Почту |
| `was_phone_lead` | Был лид по телефону |

#### C. **Категориальные признаки (postman — email-коммуникации)**

| Признак | Описание |
|---------|----------|
| `mm_event_30`, `mm_event_90` | Маркетинговое событие |
| `postman_action_30`, `postman_action_90` | Действие (open, click, smtp_open) |
| `postman_campaign_30`, `postman_campaign_90` | ID кампании |
| `postman_response_30`, `postman_response_90` | Результат (ok, fail) |
| `postman_scenario_30`, `postman_scenario_90` | Сценарий рассылки |

---

### 6. **Мета-признаки**

| Признак | Тип | Описание |
|---------|-----|----------|
| `profile_id` | string | Уникальный идентификатор профиля |
| `entity_id` | string | Идентификатор реального человека (целевая переменная) |
| `entity_type` | string | `single_profile` или `multi_profile` |
| `is_multi_entity` | bool | Производный признак (True для multi_profile) |

